# Trích feature (cache) trên Kaggle

Trích feature của các encoder đã chọn **một lần** cho mỗi part; `train.ipynb` chỉ đọc cache này, không cần encoder/worker.

| | |
|---|---|
| **Encoder** | `fate` (B-FATE), `avhubert` (B-AVH, P2 và các biến thể), `dinov2` (nhánh artifact P2): chọn 1, nhiều hoặc `"all"` |
| **Thời gian part1 (2×T4)** | FATE ~5,6 h · AV-HuBERT ~4,7 h · DINOv2 ~0,4 h. Cả 3 sát giới hạn 12 h: nên tách FATE và AV-HuBERT+DINOv2 ra 2 phiên |
| **Đầu ra** | `vn-av-df-forensics/cache/vn-av-df-data/<encoder>/` (cùng đường dẫn local). Output chỉ giữ thư mục cache của encoder đã chọn |
| **Kaggle** | Bật Internet + GPU. Worker AV-HuBERT chỉ cài khi chọn `avhubert` |

## 1. Cài đặt
Clone repository, cài `data_pipeline` và gói `[training]`.

In [ ]:
from pathlib import Path
import subprocess, sys, os
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-df-forensics.git"  # Sửa nếu đổi repository.
ROOT = Path("/kaggle/working/vn-av-df-forensics")
# Môi trường worker (~10+ GB) để ngoài /kaggle/working: không tính vào Output 20 GB, mất khi hết phiên.
WORKERS = Path("/kaggle/temp/vn-av-df-workers")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
assert (ROOT / "settings.py").is_file(), "Cần upload/push code mới trước khi dùng notebook"
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT / "data_pipeline"), "-e", str(ROOT) + "[training]"], check=True)
import settings
settings.bootstrap()
from vn_av_df.actions import execute

## 2. Chọn part và encoder
- `DATA_PARTS`: đường dẫn đầy đủ từng part (Kaggle dataset `vn-av-df-data-partN`); cache gắn với hash dữ liệu nên train phải dùng đúng các part này.
- `ENCODERS`: `"all"` hoặc danh sách trong `fate`, `avhubert`, `dinov2`. DINOv2 cắt miệng theo hộp của cache AV-HuBERT: chọn cùng `avhubert`, hoặc có `avhubert/` trong `CACHE_INPUTS`.
- `CACHE_INPUTS`: cache đã có (dataset `vn-av-df-features-partN` hoặc `.../vn-av-df-forensics/cache/vn-av-df-data` của Output phiên trước); mỗi đường dẫn là thư mục chứa `fate/`, `avhubert/`, `dinov2/`. Encoder đang chọn được chép để trích tiếp (mẫu đã có bỏ qua); encoder khác chỉ để đọc, không vào Output.

In [ ]:
# Mỗi part một đường dẫn đầy đủ (copy từ panel Input): thư mục có dataset_info.json,
# hoặc gốc dataset chứa đúng một thư mục như vậy. Nhiều part: liệt kê hết.
DATA_PARTS = [Path("/kaggle/input/datasets/YOUR_USER/vn-av-df-data-part1")]

ENCODERS = "all"  # fate + avhubert + dinov2.
# ENCODERS = ["fate"]  # Chỉ B-FATE.
# ENCODERS = ["avhubert", "dinov2"]  # B-AVH, P2 và các biến thể P2.
# ENCODERS = ["dinov2"]  # Chỉ DINOv2; cần avhubert/ trong CACHE_INPUTS.
DEVICE = "cuda"
CACHE_INPUTS = []  # Ví dụ [Path("/kaggle/input/datasets/YOUR_USER/vn-av-df-features-part1")].
CACHE_DIR = ROOT / "cache/vn-av-df-data"  # Output (cùng đường dẫn local).

## 3. Thông số trích feature
Đổi thông số làm đổi signature cache: mẫu cũ bị trích lại và checkpoint train trên cache cũ không dùng chung được.
`batch_windows`, `workers_per_device`, `batch_size` chỉ đổi tốc độ/VRAM, không đổi feature.

In [ ]:
FATE_FEATURES = {
    "precision": "float16",  # CPU: đổi thành float32.
    "window_s": 2.0, "step_s": 0.2, "temporal_bins": 8,
    "min_coverage": 0.75, "max_duration": 60, "max_side": 640,
    "batch_windows": 8,      # Số cửa sổ mỗi lượt model; chỉ đổi tốc độ/VRAM. Thiếu VRAM: giảm.
}
AVH_FEATURES = {
    "chunk_frames": 200, "overlap_frames": 50,
    "output_stride": 5,      # 25 Hz / 5 = output 5 Hz, mỗi ô 0.2 s.
    "max_duration": 60, "max_side": 640,
    "workers_per_device": 2,  # Worker mỗi GPU: dlib dò mặt chạy CPU một luồng (Kaggle 4 CPU, 2 GPU).
}
DINO_FEATURES = {
    "precision": "float16",  # CPU: đổi thành float32.
    "crop_size": 224,        # Crop miệng RGB cho DINOv2 ViT-S/14, bội số 14.
    "batch_size": 64,        # Giảm nếu thiếu VRAM.
}

## 4. Áp cấu hình, gom cache cũ, tải encoder và worker
Kiểm tra dữ liệu (hash từng part), tải encoder đã chọn; worker AV-HuBERT chỉ cài khi chọn `avhubert`.

In [ ]:
ALL_ENCODERS = ["fate", "avhubert", "dinov2"]
# Mỗi encoder trích qua một kiến trúc dùng nó; prepare_encoders giới hạn đúng encoder đã chọn.
VIA = {"fate": "fate_gru", "avhubert": "avh_tcn", "dinov2": "p2_syncartifact"}
chosen = ALL_ENCODERS.copy() if ENCODERS == "all" else ENCODERS
if not isinstance(chosen, list) or not chosen or len(set(chosen)) != len(chosen) or set(chosen) - set(ALL_ENCODERS):
    raise ValueError(f"ENCODERS phải là 'all' hoặc danh sách không trùng trong {ALL_ENCODERS}")
encoders = [e for e in ALL_ENCODERS if e in chosen]  # AV-HuBERT luôn trích trước DINOv2.
from vn_av_df.dataset import part_selection
settings.DATASET, settings.DATASET_PARTS = part_selection(DATA_PARTS)
settings.ARCHITECTURES, settings.DEVICE = [VIA[e] for e in encoders], DEVICE
cfg = settings.config()
cfg["cache"] = str(CACHE_DIR)
cfg["prepare_encoders"] = encoders
cfg["encoders"]["fate"].update(FATE_FEATURES)
cfg["encoders"]["avhubert"].update(AVH_FEATURES)
cfg["encoders"]["dinov2"].update(DINO_FEATURES)
import torch
# Trích feature song song trên mọi GPU (mỗi GPU một encoder); cache giống nhau dù trích ở GPU nào.
cfg["prepare_devices"] = [f"cuda:{i}" for i in range(torch.cuda.device_count())] if DEVICE == "cuda" else None
from vn_av_df.experiment import attach_caches
reused = attach_caches(CACHE_INPUTS, CACHE_DIR, writable=encoders)
if "dinov2" in encoders and "avhubert" not in encoders and "avhubert" not in reused:
    raise ValueError("DINOv2 cần cache AV-HuBERT: thêm 'avhubert' vào ENCODERS hoặc CACHE_INPUTS")
from vn_av_df.dataset import training_dataset
rows, selection = training_dataset(cfg, verify_media=True)
print("Part và hash:", selection)
print("Encoder:", encoders, "| Cache cũ:", {k: str(v) for k, v in reused.items()}, "| Output:", cfg["cache"])
execute("encoder_setup", cfg)  # Chỉ tải encoder đã chọn.

In [ ]:
# DINOv2 không cần worker: hộp miệng đọc từ cache AV-HuBERT.
needs_avhubert = "avhubert" in cfg["prepare_encoders"]
if needs_avhubert:
    try:
        subprocess.run([sys.executable, str(ROOT / "environments/bootstrap_worker.py"), "avhubert", "--root", str(ROOT), "--env-root", str(WORKERS)], check=True)
    except subprocess.CalledProcessError:
        # Lỗi thật (compiler, pip) nằm trong log cài đặt, không nằm trong CalledProcessError.
        log = WORKERS / ".kaggle-tools/avhubert/setup.log"
        if log.exists():
            lines = log.read_text(encoding="utf-8", errors="replace").splitlines()
            hits = [i for i, line in enumerate(lines) if "error" in line.lower()]
            start = max(0, hits[0] - 5) if hits else len(lines) - 80
            print("\n".join(lines[start:start + 60]))  # Đoạn quanh lỗi đầu tiên.
            print("...\n" + "\n".join(lines[-20:]))
        raise RuntimeError(f"Cài worker AV-HuBERT lỗi; xem đoạn log ở trên ({log})") from None
    worker_python = str(WORKERS / ".venv-avhubert/bin/python")
    settings.AVH_PYTHON = worker_python
    cfg["encoders"]["avhubert"]["python"] = worker_python
    print("Worker ready:", worker_python)
else:
    print("Không trích AV-HuBERT: bỏ qua worker.")

## 5. W&B (tuỳ chọn)
Run `prepare-<encoder>-<ngày giờ>` (group `prepare`): dữ liệu đã chọn (số mẫu theo split × ô 2×2, theo speaker), log console, GPU/RAM và thời gian từng encoder.
- Lần đầu: thêm Kaggle secret `WANDB_API_KEY` (*Add-ons → Secrets*, bật cho notebook).
- Tắt: đặt `USE_WANDB = False`. Không có secret thì tự tắt, pipeline vẫn chạy.

In [ ]:
import json
USE_WANDB = True
WANDB_PROJECT = "vn-av-df-forensics"
WANDB_ENTITY = None  # Team W&B; None = tài khoản của API key.
if USE_WANDB:
    # Log W&B ở /kaggle/working/wandb (ngoài repo): còn trong Output, `wandb sync` lại được.
    os.environ.setdefault("WANDB_DIR", "/kaggle/working")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb
    try:
        from kaggle_secrets import UserSecretsClient
        # Key qua biến môi trường (không truyền vào code); W&B tự đọc.
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
        wandb.login()
    except Exception as exc:  # Không có secret: tắt W&B, pipeline vẫn chạy.
        USE_WANDB = False
        print("Tắt W&B, không đọc được secret WANDB_API_KEY:", exc)

def plain(value):
    """Cấu hình/metric về kiểu JSON đơn giản để W&B nhận."""
    return json.loads(json.dumps(value, default=str))

def stamp():
    """Ngày giờ Việt Nam (máy Kaggle chạy UTC) gắn vào tên run để phân biệt các lần chạy."""
    from datetime import datetime, timedelta, timezone
    return datetime.now(timezone(timedelta(hours=7))).strftime("%m%d-%H%M")

def table(rows):
    """Bảng W&B từ list dict; ô lồng (dict/list) đổi thành chuỗi JSON."""
    import pandas as pd
    frame = pd.DataFrame(rows)
    for column in frame.columns:
        if frame[column].map(lambda v: isinstance(v, (dict, list))).any():
            frame[column] = frame[column].map(lambda v: json.dumps(v, ensure_ascii=False))
    return wandb.Table(dataframe=frame)

def dataset_tables():
    """Dữ liệu đã chọn: số mẫu theo split × ô 2×2 và theo split × speaker (từ rows của mục 4)."""
    from collections import Counter
    from vn_av_df.dataset import condition
    cells = Counter((r["split"], condition(r)) for r in rows)
    speakers = {}
    for r in rows:
        item = speakers.setdefault((r["split"], r.get("speaker_id") or "-"), {"samples": 0, "minutes": 0.0, "sources": set()})
        item["samples"] += 1
        item["minutes"] += r["duration_s"] / 60
        item["sources"].add(r["source_id"])
    return {
        "data/by_condition": table([{"split": s, "condition": c, "samples": n} for (s, c), n in sorted(cells.items())]),
        "data/by_speaker": table([{"split": s, "speaker_id": p, "samples": v["samples"], "sources": len(v["sources"]),
                                    "minutes": round(v["minutes"], 2)} for (s, p), v in sorted(speakers.items())]),
    }

def start_prepare_run():
    """Run riêng cho trích feature: W&B ghi log console và GPU/RAM suốt bước prepare."""
    if not USE_WANDB:
        return None
    if wandb.run is not None:
        wandb.finish()
    run = wandb.init(
        project=WANDB_PROJECT, entity=WANDB_ENTITY, job_type="prepare", group="prepare",
        name=f"prepare-{'-'.join(encoders)}-{stamp()}",
        config=plain({"dataset": selection, "encoders": encoders,
                      "features": {"fate": FATE_FEATURES, "avhubert": AVH_FEATURES, "dinov2": DINO_FEATURES}}),
    )
    run.log(dataset_tables())
    return run

def finish_prepare_run(run):
    """Thời gian trích theo encoder (extraction.json của từng cache) rồi đóng run."""
    if run is None:
        return
    cache = Path(cfg["cache"])  # Prepare lỗi sớm có thể chưa tạo thư mục cache.
    for folder in sorted(cache.glob("*/extraction.json")) if cache.is_dir() else []:
        folder = folder.parent
        info = json.loads((folder / "extraction.json").read_text(encoding="utf-8"))
        run.summary.update({f"{folder.name}/samples": info["samples"],
                            f"{folder.name}/minutes": round(info["elapsed_s"] / 60, 2),
                            f"{folder.name}/seconds_per_sample": round(info["elapsed_s"] / max(1, info["samples"]), 2)})
    run.finish()

## 6. Trích feature
- Mẫu đã có cache đúng signature được bỏ qua: phiên bị ngắt thì attach Output, thêm `.../vn-av-df-forensics/cache/vn-av-df-data` vào `CACHE_INPUTS` và chạy lại.
- Có trích feature test nhưng không fit thống kê hay train trên test.

In [ ]:
# W&B: run "prepare-<encoder>". Tắt W&B hoặc bỏ qua cell mục 5 thì không ghi, prepare vẫn chạy.
prepare_run = start_prepare_run() if globals().get("USE_WANDB") else None
try:
    execute("prepare", cfg)
finally:
    if prepare_run is not None:
        finish_prepare_run(prepare_run)

## 7. Dọn Output
- Giữ `cache/vn-av-df-data/<encoder>/` của encoder đã chọn (kèm `extraction.json`: số mẫu, thời gian, GPU) và thông tin worker AV-HuBERT (`cache/vn-av-df-data/environment/avhubert/`); xoá code, source, weights và symlink cache chỉ đọc.
- Dataset tạo thẳng từ Output bị giới hạn **500 file** (cache có ~6000 file/encoder): tải Output về (`tools/kaggle_pull.py`), zip các thư mục encoder rồi upload thành dataset `vn-av-df-features-partN` (README mục 3). `train.ipynb` trỏ `CACHE_INPUTS` tới dataset đó.

In [ ]:
import json, shutil
from vn_av_data.common.runtime import prune_output
cache = Path(cfg["cache"])
keep = [(cache / key).relative_to(ROOT) for key in encoders]
# Thông tin worker (receipt, pip freeze, log cài) đi cùng cache; WORKERS ở /kaggle/temp sẽ mất khi hết phiên.
if needs_avhubert:
    target = cache / "environment/avhubert"
    target.mkdir(parents=True, exist_ok=True)
    for p in (WORKERS / ".kaggle-tools/avhubert").glob("*"):
        if p.is_file():
            shutil.copy2(p, target / p.name)
    keep.append(target.relative_to(ROOT))
prune_output(ROOT, keep)

def size_gb(path):
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 1e9 if path.exists() else 0.0

for key in encoders:
    info = cache / key / "extraction.json"
    done = json.loads(info.read_text(encoding="utf-8")) if info.exists() else {}
    print(f"{key}: {size_gb(cache / key):.2f} GB, {done.get('samples', 'chưa xong')} mẫu, "
          f"{done.get('elapsed_s', 0) / 3600:.2f} h")
print(f"Tổng /kaggle/working: {size_gb(Path('/kaggle/working')):.2f} GB (giới hạn Output 20 GB)")